# Quick check — Task 1: MAP selection coefficients over the full window

Runs the MAP estimation (`src/map_estimation/`) on a synthetic population with known selection coefficients:

1. build the data matrix from a synthetic FASTA with `FluStudy`;
2. fit the MPL selection coefficients with `MAPestimate` using every time point in the window;
3. keep the alleles seen above 1% at two consecutive time points (`map_selection_table`), as `run_map_estimation.jl` does.

Runtime: under a minute, plus compilation on the first run.

In [ ]:
# Run from the notebooks/ folder (or the project root)
const PROJECT_ROOT = isfile(joinpath(pwd(), "Project.toml")) ? pwd() : normpath(joinpath(pwd(), ".."))
using Pkg; Pkg.activate(PROJECT_ROOT)
using Dates, DataFrames, Statistics, CSV

include(joinpath(PROJECT_ROOT, "src", "shared", "mpl_inference.jl"))        # MAPestimate, intToaa, ...
include(joinpath(PROJECT_ROOT, "src", "shared", "FluStudy.jl"))
include(joinpath(PROJECT_ROOT, "src", "map_estimation", "map_selection.jl"))  # map_selection_table
include(joinpath(PROJECT_ROOT, "src", "shared", "synthetic_data.jl"))
include(joinpath(PROJECT_ROOT, "src", "shared", "cli.jl"))                    # run_name (output file names)
using .SyntheticFlu
const AA = "-ACDEFGHIKLMNPQRSTVWY"                   # integer state -> amino-acid letter

## 1. Synthetic input

`simulate_population` (`src/shared/synthetic_data.jl`) evolves a population with the negative-binomial dynamics of `src/shared/population_simulation.jl`, using its default parameters:

* 20 sites with 4 states each; selection coefficients `s ~ Normal(0, 0.05)`, with 0 for the alleles of a random reference sequence;
* 100 random starting sequences sharing 1000 individuals;
* offspring numbers negative binomial with R = 2 and dispersion k = 0.1, population ceiling 10 000;
* mutation rate 10⁻³ per site per generation;
* 1000 generations, one generation per day (2009-01-02 … 2011-09-28).

Each day 20 individuals are sampled and "sequenced" (states 1–4 are written as the amino acids A, C, D, E). The samples are saved as a FASTA file with the same header layout as the real data, then read back with `FluStudy.create_study`, as the command-line scripts do (`protein_length = 20` instead of 566 for HA).

In [ ]:
syn = simulate_population(seed = 1, n_sample = 20)
fasta = joinpath(mktempdir(), "synthetic_H3N2_HA.fasta")
write_fasta(fasta, syn)

start_date, stop_date = Date(2009, 1, 1), Date(2009, 1, 1) + Day(1000)
study = redirect_stdout(devnull) do
    FluStudy.create_study(fasta; protein = "HA", protein_length = syn.L,
                          start_date = start_date, stop_date = stop_date, tstep = 10.0, tsmooth = 60.0)
end
data = study.data
println("population size: median ", round(Int, median(syn.pop_size)), ", range ", extrema(syn.pop_size),
        " | distinct sequences in the population: median ", round(Int, median(syn.n_species)))
println("selection coefficients: ", count(>(0.05), syn.s), " alleles with s > 0.05, ", count(<(-0.05), syn.s), " with s < -0.05 (of ", length(syn.s), ")")

In [ ]:
# FluStudy reproduces the synthetic samples: the same (day, count, sequence) rows, up to their order
# within a day. It drops the last FASTA date, because its daily bins are right-open.
sorted_rows(m) = sort(Tuple.(eachrow(m)))
@assert sorted_rows(data) == sorted_rows(syn.data[syn.data[:, 1] .< maximum(syn.data[:, 1]), :])
println("data matrix: ", size(data), "  (rows × [day, count, $(syn.L) sites])")

## 2. MAP estimate over the full window

The same call as `run_map_estimation.jl`: the fit runs up to the stop date, so all time points are used (L2 regularisation `gamma = 20`, 10-day grid, 60-day Gaussian smoothing). The inferred coefficients are relative to the reference (the consensus at the first time point), so the true ones are re-expressed relative to the same reference before comparing.

With these dynamics (strong drift from the overdispersed offspring numbers, linkage, recurrent mutation) the correlation between inferred and true coefficients was 0.66–0.85 across 20 seeds in our tests.

In [ ]:
map_result = redirect_stdout(devnull) do
    MAPestimate(data, stop_date, "H3N2"; tstep = 10.0, tsmooth = 60.0, gamma = 20.0, q = 21, correlation = true)
end
q = 21
s_true = normalize_to_reference(syn.s21, map_result.ref_s)
seen = [a != map_result.ref_s[i] && maximum(map_result.observed_path_freq[(i - 1) * q + a, :]) > 0.01
        for i in 1:syn.L for a in 1:q]                     # non-reference alleles seen above 1%
comparison = DataFrame(site = [i for i in 1:syn.L for a in 1:q][seen],
                mutation = [string(AA[map_result.ref_s[i]], i, AA[a]) for i in 1:syn.L for a in 1:q][seen],
                s_true = round.(s_true[seen], digits = 3), sc_MAP = round.(map_result.h[seen], digits = 3))
sort(comparison, :s_true, rev = true)

In [ ]:
# Sanity checks: the inferred coefficients follow the true ones
r = cor(map_result.h[seen], s_true[seen])
strong = seen .& (abs.(s_true) .> 0.05)
agree = count(sign.(map_result.h[strong]) .== sign.(s_true[strong])) / count(strong)
println("correlation(true, MAP) = ", round(r, digits = 3), " over ", count(seen), " alleles; sign agreement for |s| > 0.05: ",
        round(agree, digits = 2), " (", count(strong), " alleles)")
@assert r > 0.5
@assert agree > 0.6
@assert all(map_result.h[(i - 1) * q + map_result.ref_s[i]] == 0 for i in 1:syn.L)   # reference alleles at 0

## 3. Selection-coefficient table

`map_selection_table` keeps the non-gap alleles whose frequency exceeds `freq_threshold` at two consecutive time points at least once. `protein = "HA"` would add an HA domain column; it is left out here because the synthetic sites are not HA positions.

In [ ]:
tbl = map_selection_table(map_result; q = q, freq_threshold = 0.01, protein = "synthetic")
first(sort(tbl, :Selection_coefficient, rev = true), 10)

In [ ]:
# Every kept allele passes the filter, and every allele that passes is kept
f = map_result.observed_path_freq
passes(k) = (a = f[k, :] .> 0.01; any(a[1:end-1] .& a[2:end]))
kept = Set((r.Position - 1) * q + aaToint[charToaa[only(r.Mutation)]] for r in eachrow(tbl))
@assert all(passes, kept)
@assert all(k -> !passes(k) || k in kept, [(i - 1) * q + a for i in 1:syn.L for a in 2:q])

out = joinpath(mktempdir(), run_name("map_selection", "H3N2", "HA", start_date, stop_date; gamma = 20.0, tsmooth = 60.0, minfreq = 0.01) * ".csv")
CSV.write(out, tbl)
println(nrow(tbl), " alleles written to ", out)